# 🔧 Build approval gates into the agent loop

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 20 §20.6 (with §20.3–§20.5) · type: walkthrough*

**The promise:** wire the risk-tier table into the loop so a gated tool call **checkpoints and parks the run as `waiting_human`**, and an approval (or denial) arriving minutes or days later **resumes the run cleanly** — denial injected as an *error-shaped tool result* so the agent adapts instead of crashing.

## 🧠 Why this matters

This is the chapter's 🔧 **Build**, and its quiet lesson is that the gate is *architecture you already had*. Checkpointed runs (Ch 14) already gave you **pause**. Tool results already gave you a **channel for injecting an outside decision** back into the model's context. The audit log was already flowing to observability (Ch 23). An approval gate is just those three pieces pointed at a human.

From the agent's point of view, a gated tool is nothing more exotic than a **slow tool** — it asks, the loop suspends, and some time later a `tool_result` shows up and the run continues. Part VII later adds only the FastAPI endpoints, the Celery worker, and the notification job; the control flow you build here is the whole idea.

## Objectives & prerequisites

**By the end you can:**
- Add reversibility-tiered approval gates to any agent loop with a tier-aware `execute`.
- Park a run as `waiting_human` (checkpoint it) and `resolve` it across an out-of-band human decision.
- Handle a **denial** by injecting an error-shaped tool result so the agent re-plans rather than crashes.

**Prereqs:** [`20-01-autonomy-tiers-and-escalation.ipynb`](./20-01-autonomy-tiers-and-escalation.ipynb) (the `Tier`/`TOOL_TIERS` table); Ch 14 (checkpoint/resume — the mechanism this reuses); Ch 12 (the loop); Ch 16 (`RunBudget`); Ch 17 (the team whose tools get gated). **Offline by default** — `MOCK=1` runs the full park→approve and park→deny cycles with a scripted mock LLM and mock tools. No outbound actions ever; the gated tools are mocks.

## Setup

`MOCK=1` (default) scripts the LLM and stubs every tool so the pause/resume cycle is deterministic and free. `MOCK=0` would run one short real turn that pauses at the gate (the gated tools stay mocks — we never send real email or move real money in a notebook).

In [ ]:
import os
import json
import random
import uuid
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # secrets come ONLY from the environment / a git-ignored .env

MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

SEED = 2002
random.seed(SEED)

DATA = Path("data")  # tiny committed fixtures live here

print(f"MOCK mode: {MOCK}")
print("Full park->approve and park->deny cycles run offline in MOCK mode — no API key, no outbound actions.")

Load the two sample tickets. The agent will work a refund request (`T-1042`) — exactly the kind of customer-visible action that must be gated.

In [ ]:
TICKETS = json.loads((DATA / "tickets.json").read_text(encoding="utf-8"))
for tid, t in TICKETS.items():
    print(f"{tid}: {t['subject']}  (order €{t['order_total_eur']})")

## The tier table (§20.6)

Same `Tier` enum and `TOOL_TIERS` map as the book and notebook 20-01. `search_docs`/`get_ticket` are READ, `create_ticket` is REVERSIBLE, `send_reply`/`refund` are **GATED**. The default for an unknown tool is **GATED** — fail safe.

In [ ]:
class Tier(str, Enum):
    READ = "read"
    REVERSIBLE = "reversible"
    GATED = "gated"  # human approval required


TOOL_TIERS = {
    "search_docs":   Tier.READ,
    "get_ticket":    Tier.READ,
    "create_ticket": Tier.REVERSIBLE,
    "send_reply":    Tier.GATED,
    "refund":        Tier.GATED,
}

## `PendingApproval` — what the approver actually sees (§20.6, §20.2)

A design rule made concrete: **an approval must carry enough context to judge** — the action, its exact arguments, the agent's stated *reasoning*, and what happens if approved. "Approve?" with a bare JSON blob is rubber-stamp manufacturing. The `tool_use_id` is what lets the eventual result rejoin the *right* call.

In [ ]:
@dataclass
class PendingApproval:
    run_id: str
    tool_use_id: str           # so the result rejoins the right call
    tool: str
    args: dict
    reasoning: str             # the agent's own justification, shown to the approver
    id: str = field(default_factory=lambda: f"pa_{uuid.uuid4().hex[:8]}")

    def render_for_human(self) -> str:
        return (
            f"APPROVAL REQUEST [{self.id}]\n"
            f"  action    : {self.tool}\n"
            f"  arguments : {self.args}\n"
            f"  reasoning : {self.reasoning}\n"
            f"  consequence: executes a GATED tool on real systems if approved"
        )

## A tiny run store + scripted mock LLM and tools

The store is the in-memory stand-in for Ch 14's checkpointer — `checkpoint` persists a run, `save_pending`/`load_pending` hold approvals. The mock LLM is *scripted* so the run deterministically reaches a gated `send_reply`, then (after the human decision rejoins as a tool result) produces a final message. Every tool here is a stub — **no real outbound action**.

In [ ]:
@dataclass
class ToolUse:
    """A model 'tool_use' block, shaped like the Anthropic SDK's."""
    id: str
    name: str
    input: dict
    type: str = "tool_use"


@dataclass
class Run:
    id: str
    messages: list = field(default_factory=list)
    status: str = "running"   # running | waiting_human | done
    step: int = 0             # which scripted model turn we are on

    def last_assistant_text(self) -> str:
        for m in reversed(self.messages):
            if m["role"] == "assistant" and isinstance(m["content"], str):
                return m["content"]
        return ""


class RunStore:
    """In-memory stand-in for the Ch 14 checkpointer + a pending-approval table."""
    def __init__(self):
        self._runs: dict[str, Run] = {}
        self._pending: dict[str, PendingApproval] = {}

    def checkpoint(self, run: Run) -> None:
        # deep-ish copy via the messages list so a later mutation can't rewrite history
        self._runs[run.id] = Run(run.id, list(run.messages), run.status, run.step)

    def load_run(self, run_id: str) -> Run:
        saved = self._runs[run_id]
        return Run(saved.id, list(saved.messages), saved.status, saved.step)

    def save_pending(self, p: PendingApproval) -> None:
        self._pending[p.id] = p

    def load_pending(self, pending_id: str) -> PendingApproval:
        return self._pending[pending_id]


AUDIT: list[dict] = []


def audit_log(run_id, tool, args, tier, result, approver=None):
    """Ch 23: who/what/why for every consequential action, human decisions included."""
    AUDIT.append({"run": run_id, "tool": tool, "args": args, "tier": str(tier),
                  "result": result, "approver": approver})

In [ ]:
def run_tool(name: str, args: dict) -> str:
    """All tools are MOCKS — deterministic, side-effect free. We never send real email/money."""
    if name == "get_ticket":
        return json.dumps(TICKETS.get(args.get("id"), {}))
    if name == "search_docs":
        return "policy: duplicate charges are refundable within 30 days [src: billing-faq#dupes]"
    if name == "send_reply":
        return f"(mock) reply sent to {args.get('to')}: {args.get('body', '')[:40]}..."
    if name == "refund":
        return f"(mock) refunded €{args.get('amount_eur')} for {args.get('ticket')}"
    return f"(mock) ran {name}"


def mock_model_turn(run: Run) -> dict:
    """Scripted assistant turns so the pause/resume cycle is deterministic.

    Turn 0: reason, then attempt a GATED send_reply.
    Turn 1+: react to whatever tool_result rejoined the conversation (approval or denial)
             and emit a final natural-language message (no more tool calls).
    """
    if run.step == 0:
        reasoning = ("The ticket shows a duplicate €49 Pro charge; policy allows a refund "
                     "within 30 days. I'll reply to confirm the duplicate refund.")
        tool_use = ToolUse(id=f"toolu_{uuid.uuid4().hex[:8]}", name="send_reply",
                           input={"to": TICKETS['T-1042']['customer'],
                                  "body": "We've refunded your duplicate €49 charge."})
        return {"reasoning": reasoning, "tool_use": tool_use}
    # After resume: read the last tool_result and decide what to say.
    last = run.messages[-1]
    tr = last["content"][0]["content"] if last["role"] == "user" else ""
    if "denied" in tr:
        text = ("Understood — I won't send that reply. I'll ask the customer to confirm "
                "the duplicate before issuing any refund, per the reviewer's note.")
    else:
        text = "Reply sent and refund confirmed. Closing the ticket."
    return {"reasoning": text, "tool_use": None}

## The tier-aware `execute` — park instead of run (§20.6)

Here is the core. On a **GATED** call the executor builds a `PendingApproval` (with the agent's reasoning), saves it, flips the run to `waiting_human`, **checkpoints** the run, and raises `NeedsApproval` *instead of executing*. READ/REVERSIBLE calls run immediately and hit the audit log.

In [ ]:
class NeedsApproval(Exception):
    def __init__(self, pending: PendingApproval):
        self.pending = pending


def execute(block: ToolUse, run: Run, store: RunStore) -> str:
    tier = TOOL_TIERS.get(block.name, Tier.GATED)   # unknown => gated
    if tier is Tier.GATED:
        pending = PendingApproval(
            run_id=run.id, tool_use_id=block.id,
            tool=block.name, args=block.input,
            reasoning=run.last_assistant_text(),
        )
        store.save_pending(pending)
        run.status = "waiting_human"
        store.checkpoint(run)                       # Ch 14: park the run
        raise NeedsApproval(pending)                # notify approver (Ch 31)
    result = run_tool(block.name, block.input)
    audit_log(run.id, block.name, block.input, tier, result)  # Ch 23
    return result

## The driver loop (Ch 12) — runs until done *or* parked

A compact agent loop: ask the model, append its turn, and if it called a tool, `execute` it (which may park the run) and feed the result back. When `execute` raises `NeedsApproval`, the loop stops and surfaces the pending request — the run is safely checkpointed.

In [ ]:
def step_run(run: Run, store: RunStore, max_steps: int = 6):
    """Drive the loop. Returns ('done', run) or ('waiting_human', PendingApproval)."""
    while run.step < max_steps:
        turn = mock_model_turn(run)
        run.messages.append({"role": "assistant", "content": turn["reasoning"]})
        run.step += 1
        block = turn["tool_use"]
        if block is None:
            run.status = "done"
            return "done", run
        try:
            result = execute(block, run, store)
        except NeedsApproval as na:
            return "waiting_human", na.pending  # run is checkpointed; loop suspends here
        # non-gated tool ran: feed its result back as a tool_result and continue
        run.messages.append({"role": "user", "content": [
            {"type": "tool_result", "tool_use_id": block.id, "content": result}]})
    run.status = "done"
    return "done", run


def continue_run(run: Run, store: RunStore):
    """Re-enter the loop after a human decision rejoined as a tool_result."""
    return step_run(run, store)

## The resume path — `resolve` injects the decision as a tool result (§20.6)

This is the elegant half. The human's decision becomes the gated tool's **result**. Approved calls execute *now*. Denials are injected as an **error-shaped** result — not a crash — so the agent can adapt (propose an alternative, ask a question). Either way the result rejoins via `tool_use_id`, the run flips back to `running`, and `continue_run` re-enters the loop.

In [ ]:
def resolve(pending_id: str, approved: bool, note: str, store: RunStore):
    p = store.load_pending(pending_id)
    run = store.load_run(p.run_id)                  # restore the checkpoint
    if approved:
        result = run_tool(p.tool, p.args)
        audit_log(run.id, p.tool, p.args, "approved", result, approver=note)
    else:
        result = (f"error: action denied by reviewer. Reason: {note}. "
                  "Adjust your approach or ask the user.")
        audit_log(run.id, p.tool, p.args, "denied", result, approver=note)
    run.messages.append({"role": "user", "content": [
        {"type": "tool_result", "tool_use_id": p.tool_use_id, "content": result}]})
    run.status = "running"
    store.checkpoint(run)
    return continue_run(run, store)                 # re-enter the loop

## Cycle 1 — park → **approve** → resume

Start a run on the duplicate-charge ticket. The agent reasons, then attempts a gated `send_reply` — which parks the run. We then inspect exactly what the human would see.

In [ ]:
store = RunStore()
run = Run(id="run_approve")
run.messages.append({"role": "user", "content": "Handle ticket T-1042 (duplicate charge)."})

state, payload = step_run(run, store)
print("loop state:", state)
print()
print(payload.render_for_human())

The run is now `waiting_human` and **checkpointed** — it could sit here for minutes or days; nothing is lost. A reviewer approves. Watch the gated tool finally execute and the run drive to completion.

In [ ]:
state, final = resolve(payload.id, approved=True, note="sami@support", store=store)
print("loop state:", state)
print("final agent message:", final.last_assistant_text())
print("audit trail:")
for row in AUDIT:
    print("  ", row["tool"], "->", row["tier"], "| approver:", row["approver"])

## Cycle 2 — park → **deny with a reason** → re-plan

### 🔮 Predict

We'll run the *same* ticket, but this time the reviewer **denies** the `send_reply` with the note *"confirm the duplicate with the customer first."* The denial is injected as an error-shaped `tool_result`. **Predict:** does the agent crash? Retry blindly? Or read the error and change course? Write your guess, then run the two cells.

In [ ]:
AUDIT.clear()
store2 = RunStore()
run2 = Run(id="run_deny")
run2.messages.append({"role": "user", "content": "Handle ticket T-1042 (duplicate charge)."})

state, pending = step_run(run2, store2)
print("loop state:", state, "| parked at:", pending.tool)

In [ ]:
state, final = resolve(
    pending.id, approved=False,
    note="confirm the duplicate with the customer first", store=store2,
)
print("loop state:", state)
print("final agent message:", final.last_assistant_text())
# the error-shaped tool_result the agent re-planned from:
for m in final.messages:
    if m["role"] == "user" and isinstance(m["content"], list):
        print("injected tool_result:", m["content"][0]["content"])

**What you just saw.** The denial did **not** raise an exception. It arrived as an ordinary `tool_result` whose text begins `error: action denied...`, and the agent treated it like any failed tool — it re-planned (ask the customer to confirm) instead of crashing or blindly resending. *That* is why denials are error-shaped results, not exceptions: the agent's existing error-recovery behavior does the work for free.

### ⚠️ Pitfall (carried from 20-01): the audit log nobody read

An audit trail that "proves a human reviewed" a disaster is worthless if the human rubber-stamped it. The fix from 20-01 still applies here: **measure approval rates and retire gates that approve everything** in favor of sampled audits. The gate's value is the *attention* it buys, not the row it writes — and prefer **making actions reversible** (a send delay with cancel, soft deletes, draft-first) over adding yet another gate.

## Interruptibility & steering (§20.4) — the human's controls

Approval gates are checkpoints the *system* initiates. The complementary controls belong to the human at any moment. **Interruptibility**: a cancel flag the loop checks each iteration — a stop is just "don't take the next step," safe because state is checkpointed after every step. **Steering**: an injected message appended to the run's history *before the next model call* — the exact mechanism chat products use for mid-stream "actually, …" corrections.

In [ ]:
def loop_with_controls(run: Run, store: RunStore, cancel_flag: dict, steer: dict | None,
                       max_steps: int = 6):
    """Same loop, plus a cancel flag (checked each iteration) and message-injection steering."""
    while run.step < max_steps:
        if cancel_flag.get("stop"):                 # interruptibility: stop NOW, state is safe
            run.status = "cancelled"
            store.checkpoint(run)
            return "cancelled", run
        if steer and steer.get("pending"):          # steering: inject before the next model call
            run.messages.append({"role": "user", "content": steer.pop("pending")})
        turn = mock_model_turn(run)
        run.messages.append({"role": "assistant", "content": turn["reasoning"]})
        run.step += 1
        if turn["tool_use"] is None:
            run.status = "done"
            return "done", run
        try:
            result = execute(turn["tool_use"], run, store)
        except NeedsApproval as na:
            return "waiting_human", na.pending
        run.messages.append({"role": "user", "content": [
            {"type": "tool_result", "tool_use_id": turn["tool_use"].id, "content": result}]})
    return "done", run


demo = Run(id="run_cancel")
demo.messages.append({"role": "user", "content": "Handle ticket T-1042."})
state, _ = loop_with_controls(demo, RunStore(), cancel_flag={"stop": True}, steer=None)
print("cancelled before first step:", state)  # a stop is just 'don't take the next step'

## Trust surfaces (§20.5) — make oversight fast *and* honest

The gate is only usable if the interface makes the agent's state visible in time to act. Three surfaces carry the weight: **transparency** (stream progress — "Searching the document base… found 4 sources"), **citations** (the `CitedAnswer` contract from Ch 18 — every factual claim carries its source), and **explainability** (expose the reasoning *before* approval). One caution: a model's self-reported reasoning is *evidence* for the human, not proof — verify what must be true with Ch 16's verification.

In [ ]:
@dataclass
class CitedAnswer:                # the Ch 18 contract, shown alongside an approval request
    text: str
    sources: list[str]


def stream_progress(steps: list[str]):
    for s in steps:
        print(f"… {s}")


stream_progress(["Reading ticket T-1042", "Searching the document base… found 1 source",
                 "Drafting reply (awaiting approval)"])
answer = CitedAnswer(text="Duplicate €49 charge is refundable within 30 days.",
                     sources=["billing-faq#dupes"])
print("\nshown to approver:", answer.text, "| sources:", answer.sources)

## 🎯 Senior lens

A senior notices that **nothing here is new infrastructure**. Checkpoints (Ch 14) already gave you pause; tool results already gave you a channel to inject a human decision; the audit log already flowed to observability (Ch 23). The approval gate is those three pieces composed — which is why it's a few dozen lines, not a subsystem. Part VII adds only the *delivery* pieces: the FastAPI endpoints approvers call, the Celery worker `continue_run` dispatches to, and the notification job. And if you build on **LangGraph** instead, its `interrupt` is this exact pattern as a framework feature (Ch 18) — the architecture transfers unchanged. The judgment call a senior actually makes: *which tools earn a gate at all*, given that every avoidable gate is approval fatigue waiting to happen.

## Recap

- A gated tool is, to the agent, just a **slow tool**: the loop suspends and a `tool_result` arrives later.
- `execute` is **tier-aware** — on GATED it builds a `PendingApproval` (with reasoning), checkpoints the run as `waiting_human`, and raises `NeedsApproval` instead of executing. Unknown tools default to GATED.
- `resolve` turns the human decision into the gated tool's **result**: approved → execute now; **denied → an error-shaped tool_result** so the agent re-plans rather than crashes.
- The `tool_use_id` rejoins the decision to the right call; the run flips to `running` and `continue_run` re-enters the loop.
- **Interruptibility** (cancel flag, checked each iteration) and **steering** (message injection) are the human's complementary controls; **trust surfaces** (streaming, citations, exposed reasoning) make oversight fast and honest.
- The gate is **architecture you already had** — checkpoints + tool results + the audit log.

## Exercises

Each task *changes* the mechanism and asks you to predict the outcome first. Solutions live in `solutions/` (Phase 2).

1. **Argument-pattern tiering.** Extend `execute` so `refund` is REVERSIBLE for `amount_eur <= 25` and GATED above it (the €10-vs-€10,000 rule). Add a scripted turn that attempts a €10 refund and predict whether it parks.
2. **Approve-with-edit.** Add an `edited_args` parameter to `resolve` so a reviewer can approve a *modified* action (e.g. tone down the reply). Predict what the audit log should record so the change is traceable.
3. **Cancel mid-run.** Using `loop_with_controls`, set `cancel_flag['stop'] = True` *after* the first model turn instead of before it. Predict where the run stops and confirm its checkpointed status is clean.
4. **Steer then resume.** Inject a steering message ("only refund if the order total exceeds €40") before the gated step and predict how the agent's reasoning — and therefore the `PendingApproval.reasoning` shown to the human — changes.

In [ ]:
# Exercise 1 — argument-pattern tiering for refund (€25 boundary).


In [ ]:
# Exercise 2 — approve-with-edit: resolve(edited_args=...) + a traceable audit entry.


In [ ]:
# Exercise 3 — cancel AFTER the first turn; confirm the checkpoint is clean.


In [ ]:
# Exercise 4 — steer before the gate; watch PendingApproval.reasoning change.


## Next

You've built the toy; here's the real one.

- **Blueprint:** the production tier-aware executor + pause/resume lives in [`../../blueprints/agent-loop/`](../../blueprints/agent-loop/), and the same gate wraps the team in [`../../blueprints/multi-agent-supervisor/`](../../blueprints/multi-agent-supervisor/).
- **Template:** the `Tier`/`TOOL_TIERS` + approval scaffold ships in [`../../templates/agent-project-starter/`](../../templates/agent-project-starter/).
- **Capstone:** this becomes `capstone-project/approvals.py` and the gate in `capstone-project/loop.py` (park as `waiting_human`, resume via injected tool result). The FastAPI approval endpoints and the Celery `continue_run` dispatch land in **Part VII**. Checkpoint: `checkpoints/ch20-approval-gates`.